# BNA 岛叶 12 亚区玻璃脑可视化(真实形态版)

- **渲染引擎**: pyvista (VTK 9.3) — PBR 材质、光滑着色、高光/环境光
- **玻璃脑**: MNI152 1mm T1 等值面 (marching cubes) + Loop 细分平滑, 半透明
- **岛叶 12 亚区**: 从 AAL 岛叶 mask 提取**真实岛叶皮层 3D 形态**(沟回起伏), 再按 BNA (Brainnetome) INS-1~6 双侧 12 亚区中心做 **Voronoi 划分**着色
- **坐标体系**: 全 MNI mm
运行环境: `brainpy` conda 环境 (pyvista, nibabel, nilearn, scikit-image, scipy)

In [ ]:
# 依赖检查
import numpy as np
import pyvista as pv
print('pyvista', pv.__version__)
from insula_morph import render_morph, make_insulacortex, bna_centers_mni, voronoi_partition
print('渲染模块加载成功')

In [ ]:
# 三视角渲染 (iso 带标签)
from insula_morph import render_morph
render_morph(alpha_brain=0.06, alpha_insula=0.45,
             output='insula_morph_iso.png', view='iso', add_labels=True)
render_morph(alpha_brain=0.06, alpha_insula=0.45,
             output='insula_morph_front.png', view='front')
render_morph(alpha_brain=0.06, alpha_insula=0.45,
             output='insula_morph_top.png', view='top')
print('3 views done')

In [ ]:
# 查看渲染结果
from IPython.display import Image, display
display(Image('images/insula_morph_iso.png'))
display(Image('images/insula_morph_front.png'))

In [ ]:
# 交互式 3D 窗口 (本地运行)
import pyvista as pv
from insula_morph import make_glass_brain, make_insulacortex, bna_centers_mni, voronoi_partition, JELLY_COLORS

brain = make_glass_brain()
insula = make_insulacortex()
centers = bna_centers_mni()
names = list(centers.keys())
sub = voronoi_partition(insula, [centers[n] for n in names], names)

pl = pv.Plotter()
pl.add_mesh(brain.subdivide(1, subfilter='loop').smooth(n_iter=10, relaxation_factor=0.1),
            color='#4A7DBF', opacity=0.06)
for i, (name, sm) in enumerate(sub.items()):
    pl.add_mesh(sm, color=JELLY_COLORS[i], opacity=0.45,
                smooth_shading=True, specular=1.0, specular_power=128)
pl.show()

## 结果

成品: `images/insula_morph_iso.png` (标签版), `insula_morph_front.png`, `insula_morph_top.png`

12 亚区: Ins_L_1~6 (左), Ins_R_1~6 (右) — 左右同名同色, 编号 1~6 分别为 红/蓝/绿/紫/橙/青